<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.2-document-processing/notebooks/exercises-6.2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.2 — Document Processing for RAG
Netsetos GenAI Course v2.0 | Module 6

PDF parsing, OCR, chunking, metadata, production pipelines, India processing


In [ ]:
# pip install pymupdf4llm pdfplumber langchain-text-splitters trafilatura -q
print('Ready for document processing')


## Ex 1: PDF → Markdown → Chunks


In [ ]:
import pymupdf4llm
from langchain_text_splitters import RecursiveCharacterTextSplitter

# PDF → clean Markdown
# md_text = pymupdf4llm.to_markdown('document.pdf', write_images=True)

# Demo with sample text
md_text = '''# Company Overview\n\nACME Corp was founded in 2020.\n\n## Financial Results\n\nRevenue grew 3% to $323M in Q2 2023.\n\n| Quarter | Revenue | Growth |\n|---------|---------|--------|\n| Q1 2023 | $314M | 2% |\n| Q2 2023 | $323M | 3% |\n'''

splitter = RecursiveCharacterTextSplitter(
    chunk_size=512, chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' ', ''])
chunks = splitter.create_documents([md_text])

for i, c in enumerate(chunks):
    print(f'Chunk {i}: {len(c.page_content)} chars')
    print(f'  {c.page_content[:80]}...')
print(f'\n{len(chunks)} chunks from {len(md_text)} chars')


## Ex 2: Table Extraction Comparison


In [ ]:
# pdfplumber — best for visible-grid tables
# import pdfplumber
# pdf = pdfplumber.open('tables.pdf')
# table = pdf.pages[0].extract_table()
# print(table)  # list-of-lists

# Visual debugging
# im = pdf.pages[0].to_image()
# im.debug_tablefinder({'vertical_strategy': 'text'})

print('Table Extraction Comparison:')
tools = [
    ('pymupdf4llm', '0.12s/page', 'Markdown tables', 'Default'),
    ('pdfplumber', '0.10s/page', '96% accuracy', 'Complex grids'),
    ('Docling', 'Moderate', '97.9% accuracy', 'Air-gapped'),
]
for name, speed, acc, best in tools:
    print(f'  {name:15s} | {speed:12s} | {acc:15s} | {best}')


## Ex 3: Chunking Strategy Benchmark


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = '''Machine learning is a subset of artificial intelligence.\nIt uses algorithms to learn patterns from data.\n\nDeep learning uses neural networks with many layers.\nTransformers are a type of deep learning architecture.\n\nRAG combines retrieval with generation.\nIt grounds LLM responses in external documents.''' * 10

strategies = {
    'recursive_256': RecursiveCharacterTextSplitter(chunk_size=256, chunk_overlap=25),
    'recursive_512': RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50),
    'recursive_1024': RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=100),
}

for name, splitter in strategies.items():
    chunks = splitter.create_documents([text])
    avg = sum(len(c.page_content) for c in chunks) / len(chunks)
    print(f'{name:20s}: {len(chunks):3d} chunks, avg {avg:.0f} chars')

print('\nBenchmark: 512 = 69% accuracy (#1 in Vecta 2026)')


## Ex 4: OCR Preprocessing


In [ ]:
import numpy as np

def preprocess_for_ocr(image_path):
    '''OCR preprocessing pipeline: rescale → grayscale → denoise → binarize → deskew'''
    import cv2
    img = cv2.imread(image_path)
    img = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    denoised = cv2.medianBlur(gray, 3)
    _, binary = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # Deskew
    coords = np.column_stack(np.where(binary > 0))
    angle = cv2.minAreaRect(coords)[-1]
    angle = -(90 + angle) if angle < -45 else -angle
    (h, w) = binary.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    return cv2.warpAffine(binary, M, (w, h), flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_REPLICATE)

print('OCR Preprocessing: rescale → gray → denoise → binarize → deskew')
print('Improves accuracy by 20-40% on poor scans')


## Ex 5: Web Content Extraction


In [ ]:
# Trafilatura — F1=0.958 (best open-source)
# import trafilatura
# url = 'https://example.com/article'
# text = trafilatura.extract(trafilatura.fetch_url(url))

# Crawl4AI — self-hosted, JS rendering
# from crawl4ai import AsyncWebCrawler
# async with AsyncWebCrawler() as crawler:
#     result = await crawler.arun(url=url)
#     print(result.markdown)

print('Web Extraction Tools:')
for tool, feat in [('Trafilatura','F1=0.958, built-in dedup'),
    ('Firecrawl','API, JS rendering, recursive crawl'),
    ('Crawl4AI','Apache 2.0, 50K stars, self-hosted')]:
    print(f'  {tool:15s}: {feat}')


## Ex 6: Metadata Enrichment


In [ ]:
from langchain.schema import Document

# Enrich chunks with metadata
def enrich_chunk(text, source, page, llm=None):
    doc = Document(
        page_content=text,
        metadata={
            'source': source,
            'page': page,
            'char_count': len(text),
            'has_table': '|' in text and '-' in text,
            'has_code': '```' in text or 'def ' in text,
        }
    )
    # LLM-based enrichment (optional)
    # if llm:
    #     topics = llm.invoke(f'Extract 3 topic keywords: {text[:200]}')
    #     doc.metadata['topics'] = topics.content
    return doc

doc = enrich_chunk('Revenue grew 3% to $323M...', 'acme_10q.pdf', 12)
print(f'Content: {doc.page_content[:40]}...')
print(f'Metadata: {doc.metadata}')
print('\nMetaRAG: LLM metadata → 82.5% vs 73.3% precision')


## Ex 7: Incremental Indexing


In [ ]:
# LangChain incremental indexing
# from langchain.indexes import SQLRecordManager, index
# record_manager = SQLRecordManager(
#     namespace='chroma/my_collection',
#     db_url='sqlite:///record_manager.sql'
# )
# record_manager.create_schema()
# stats = index(docs, record_manager, vectorstore,
#               cleanup='incremental', source_id_key='source')
# print(stats)  # {num_added: 42, num_updated: 3, num_skipped: 155, num_deleted: 1}

print('Incremental Indexing Modes:')
for mode, desc in [('None','Manual cleanup only'),
    ('incremental','Removes stale versions continuously'),
    ('full','Removes all not in latest batch')]:
    print(f'  {mode:15s}: {desc}')
print('\nWithout this: re-running creates duplicates!')


## Ex 8: India Document Processing


In [ ]:
import re, unicodedata

def classify_hindi_pdf(filepath):
    '''Classify Hindi PDF: unicode, legacy font, or scanned'''
    import fitz
    doc = fitz.open(filepath)
    text = doc[0].get_text()
    devanagari = re.findall(r'[\u0900-\u097F]', text)
    if len(text.strip()) < 10:
        return 'scanned'       # → Sarvam Vision OCR
    elif len(devanagari) == 0 and any(c in text for c in 'Ùkgk'):
        return 'legacy_font'   # → kru2uni converter
    return 'native'            # → pymupdf4llm

def normalize_devanagari(text):
    '''Dual normalization for Hindi text'''
    text = unicodedata.normalize('NFC', text)
    # Strip zero-width characters
    for zw in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        text = text.replace(zw, '')
    # Indic NLP Library for script-specific fixes
    # from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
    # normalizer = IndicNormalizerFactory().get_normalizer('hi')
    # text = normalizer.normalize(text)
    return ' '.join(text.split())

hindi = 'मशीन लर्निंग एक कृत्रिम बुद्धिमत्ता है'
print(f'Original: {hindi}')
print(f'Normalized: {normalize_devanagari(hindi)}')
print(f'\nLegacy font detection: check ASCII-to-Devanagari ratio')
print('180+ non-standard fonts exist (Kruti Dev, Chanakya, Shivaji01)')


---
## Recap
Document processing = 80% of RAG quality. PDF parsing: PyMuPDF4LLM as default → tiered escalation (Docling/Marker → VLMs). OCR: preprocess first (20-40% improvement), PaddleOCR v5 for free, Sarvam Vision for Indian scripts. Chunking: recursive 512 tokens (#1 benchmark), contextual chunking for high-value docs (-49% failures at $1.02/M tokens). Metadata enrichment is the most underrated improvement (82.5% vs 73.3% precision). Production: async + incremental indexing + dead letter queues. India: three-way PDF classification (Unicode/legacy/scanned) + Devanagari normalization (NFC + Indic NLP). ColPali: future direction — vision-first document understanding.
